# vulcan_25_refactor v1

Refactor of the vulcan_2.5 training pipeline

> This document is generated from a minerva spec. Edit the spec and rebuild; changes made here are lost on the next build.

Source: `/home/tdeibert/Projects/Nuclear_Scaling/notebooks/model_training/vulcan_25_refactor.minerva.yaml` · spec `2317d6f546eb` · built 2026-09-18 18:23 UTC

## Required Packages

In [1]:
import numpy as np
from dataclasses import dataclass, field
from pathlib import Path
import os, math, time, gc, sys, json, hashlib 
import multiprocessing
import numpy
import pandas
import tifffile as tiff
import matplotlib as plt
from matplotlib.colors import ListedColormap
from scipy import ndimage
from skimage import filters, morphology, measure, segmentation, draw as skdraw
from skimage.morphology import h_maxima
from skimage.feature import peak_local_max
from sklearn.cluster import DBSCAN
from skimage.segmentation import watershed
from skimage.morphology import disk as _disk
from scipy.ndimage import distance_transform_edt, affine_transform, gaussian_filter
import roifile
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow      :", tf.__version__)
print("Num GPUs        :", len(tf.config.list_physical_devices("GPU")))
print("CPU count       :", os.cpu_count())

2026-09-18 13:28:28.152188: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-18 13:28:28.437985: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-18 13:28:28.438083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-18 13:28:28.465810: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-18 13:28:28.538183: I tensorflow/core/platform/cpu_feature_guar

TensorFlow      : 2.15.1
Num GPUs        : 1
CPU count       : 20


2026-09-18 13:28:34.871698: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-09-18 13:28:35.084499: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-09-18 13:28:35.087270: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

## Pipeline and Exp Config

In [ ]:
# === node: Config ===
# Pipeline and Exp Config
# in:
#   dataclass: package  <- Required_Packages
#   os: unspecified  <- Required_Packages
#   sys: unspecified  <- Required_Packages
#   roifile: unspecified  <- Required_Packages
#   cv2: unspecified  <- Required_Packages
# out:
#   pipe_cfg: unspecified  -> Loading, Patch_Extraction, image_features
#   exp_cfg: unspecified  -> Loading, Patch_Extraction, image_features

@dataclass 
class PipelineConfig:
    #----Pathing----#
    data_root: Path = Path("/home/tdeibert/Projects/Nuclear_Scaling/")
    model_root: Path = Path("/home/tdeibert/Projects/Nuclear_Scaling/models/")
    out_root: Path = Path("/data/user/tdeibert/Nuclear_Scaling/")



pipe_cfg = PipelineConfig()

if pipe_cfg.out_root.is_dir():
    print(f"out_root is set to {pipe_cfg.out_root}")
else:
    print(f"out_root NOT found at {pipe_cfg.out_root}")


@dataclass
class ExperimentConfig:
    #----Experiment Naming----#
    exp_name: str = "Control_1_2030_27_07"
    image_name: str = "control_extract_1.tif"
    model_version: str = "vulcan_2.5"
    

exp_cfg = ExperimentConfig()

out_root is set to /data/user/tdeibert/Nuclear_Scaling


In [ ]:
# === node: Loading ===
# Loading
# in:
#   pipe_cfg: unspecified  <- Config
#   exp_cfg: unspecified  <- Config
#   Path: package  <- Required_Packages
#   JSON: unspecified  <- Required_Packages
#   numpy: unspecified  <- Required_Packages
#   pandas: unspecified  <- Required_Packages
#   tiff: unspecified  <- Required_Packages
# out:
#   nd_array_memmap_image: unspecified  -> Patch_Extraction, image_features
#   fiji_rois: unspecified  -> Patch_Extraction
#   data_csvs: unspecified  -> Patch_Extraction
''' This cell is responsible for all loading conditions. It will load images, path objects, rois. This cell should consume cfg objects and output either a nd array, data frame or roi set'''

image = tiff.memmap(cfg.data_root/exp_cfg.exp_name/exp_cfg.image_name.tif)

## image features

In [ ]:
# === node: image_features ===
# image features
# in:
#   pipe_cfg: unspecified  <- Config
#   exp_cfg: unspecified  <- Config
#   nd_array_memmap_image: unspecified  <- Loading
#   filters: unspecified  <- Required_Packages
#   morphology: unspecified  <- Required_Packages
#   measure: unspecified  <- Required_Packages
#   segmentation: unspecified  <- Required_Packages
#   skdraw: unspecified  <- Required_Packages
#   h_maxima: unspecified  <- Required_Packages
#   peak_local_max: unspecified  <- Required_Packages
#   DBSCAN: unspecified  <- Required_Packages
#   watershed: unspecified  <- Required_Packages
#   _disk: unspecified  <- Required_Packages
#   distance_transform_edt: function  <- Required_Packages
#   affine_transform: unspecified  <- Required_Packages
#   gaussian_filter: unspecified  <- Required_Packages
#   ndimage: unspecified  <- Required_Packages
#   numpy: unspecified  <- Required_Packages
#   pandas: unspecified  <- Required_Packages
#   tiff: unspecified  <- Required_Packages
# out:
#   image_features_out: unspecified  -> Patch_Extraction
''' this section will determine the functions required for feature extraction from the images for training patch extraction'''

## Patch Extraction

In [ ]:
# === node: Patch_Extraction ===
# Patch Extraction
# in:
#   pipe_cfg: unspecified  <- Config
#   exp_cfg: unspecified  <- Config
#   nd_array_memmap_image: unspecified  <- Loading
#   fiji_rois: unspecified  <- Loading
#   data_csvs: unspecified  <- Loading
#   image_features_out: unspecified  <- image_features
# out:
#   training_patches: unspecified  -> Model_Training
raise NotImplementedError(  # stub not written yet
    'Patch_Extraction'
)

## Model Training

In [ ]:
# === node: Model_Training ===
# Model Training
# in:
#   training_patches: unspecified  <- Patch_Extraction
# out:
#   vulcan_model_number: unspecified  -> terminal
#   hist: unspecified  -> training_history
vulcan_model_number
hist

## training history

In [ ]:
# === node: training_history ===
# training history
# in:
#   hist: unspecified  <- Model_Training
# out:
#   training_history_out: unspecified  -> terminal
#   history_plots: unspecified  -> terminal
history_plots